# Statistical Analysis

In [ ]:
import pandas as pd
import numpy as np
import ast
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [ ]:
initial_df = pd.read_parquet("../data/processed/anime_data_2.parquet")
initial_df

df = initial_df.head(5278)
df.info()

## Genre, Theme, Demographic vs. Score Z-score (Multiple Regression)

In [ ]:
genre_averages = df.explode('genres').groupby('genres')['score_z'].mean().sort_values()
genre_counts = df['genres'].explode().value_counts()
print(genre_counts)
genre_averages

In [ ]:
genre_cols = [c for c in df.columns if c.startswith('genre_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_genres = [c for c in genre_cols if df[c].sum() >= min_count]
dropped = set(genre_cols) - set(valid_genres)
if dropped:
    print(f"Dropping {len(dropped)} rare genres from regression: {sorted(dropped)}")

X = df[valid_genres].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['score_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'genre': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("genre != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

In [ ]:
theme_cols = [c for c in df.columns if c.startswith('theme_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_themes = [c for c in theme_cols if df[c].sum() >= min_count]
dropped = set(theme_cols) - set(valid_themes)
if dropped:
    print(f"Dropping {len(dropped)} rare themes from regression: {sorted(dropped)}")

X = df[valid_themes].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['score_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'theme': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("theme != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

In [ ]:
demo_cols = [c for c in df.columns if c.startswith('demo_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_demos = [c for c in demo_cols if df[c].sum() >= min_count]
dropped = set(demo_cols) - set(valid_demos)
if dropped:
    print(f"Dropping {len(dropped)} rare demographics from regression: {sorted(dropped)}")

X = df[valid_demos].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['score_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'demographic': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("demographic != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

## Genre, Theme, Demographic vs. WC Z-score (Multiple Regression)

In [ ]:
genre_cols = [c for c in df.columns if c.startswith('genre_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_genres = [c for c in genre_cols if df[c].sum() >= min_count]
dropped = set(genre_cols) - set(valid_genres)
if dropped:
    print(f"Dropping {len(dropped)} rare genres from regression: {sorted(dropped)}")

X = df[valid_genres].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['wc_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'genre': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("genre != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

In [ ]:
theme_cols = [c for c in df.columns if c.startswith('theme_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_themes = [c for c in theme_cols if df[c].sum() >= min_count]
dropped = set(theme_cols) - set(valid_themes)
if dropped:
    print(f"Dropping {len(dropped)} rare themes from regression: {sorted(dropped)}")

X = df[valid_themes].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['wc_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'theme': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("theme != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

In [ ]:
demo_cols = [c for c in df.columns if c.startswith('demo_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_demos = [c for c in demo_cols if df[c].sum() >= min_count]
dropped = set(demo_cols) - set(valid_demos)
if dropped:
    print(f"Dropping {len(dropped)} rare demographics from regression: {sorted(dropped)}")

X = df[valid_demos].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['wc_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'demographic': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("demographic != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

## Genre, Theme, Demographic vs. Favorites Z-score (Multiple Regression)

In [ ]:
genre_cols = [c for c in df.columns if c.startswith('genre_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_genres = [c for c in genre_cols if df[c].sum() >= min_count]
dropped = set(genre_cols) - set(valid_genres)
if dropped:
    print(f"Dropping {len(dropped)} rare genres from regression: {sorted(dropped)}")

X = df[valid_genres].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['favorites_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'genre': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("genre != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

In [ ]:
theme_cols = [c for c in df.columns if c.startswith('theme_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_themes = [c for c in theme_cols if df[c].sum() >= min_count]
dropped = set(theme_cols) - set(valid_themes)
if dropped:
    print(f"Dropping {len(dropped)} rare themes from regression: {sorted(dropped)}")

X = df[valid_themes].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['favorites_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'theme': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("theme != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

In [ ]:
demo_cols = [c for c in df.columns if c.startswith('demo_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_demos = [c for c in demo_cols if df[c].sum() >= min_count]
dropped = set(demo_cols) - set(valid_demos)
if dropped:
    print(f"Dropping {len(dropped)} rare demographics from regression: {sorted(dropped)}")

X = df[valid_demos].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['favorites_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'demographic': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("demographic != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

## Genre, Theme, Demographic vs. Drop Rate Z-score (Multiple Regression)

In [ ]:
genre_cols = [c for c in df.columns if c.startswith('genre_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_genres = [c for c in genre_cols if df[c].sum() >= min_count]
dropped = set(genre_cols) - set(valid_genres)
if dropped:
    print(f"Dropping {len(dropped)} rare genres from regression: {sorted(dropped)}")

X = df[valid_genres].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['drop_rate_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'genre': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("genre != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

In [ ]:
theme_cols = [c for c in df.columns if c.startswith('theme_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_themes = [c for c in theme_cols if df[c].sum() >= min_count]
dropped = set(theme_cols) - set(valid_themes)
if dropped:
    print(f"Dropping {len(dropped)} rare themes from regression: {sorted(dropped)}")

X = df[valid_themes].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['drop_rate_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'theme': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("theme != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

In [ ]:
demo_cols = [c for c in df.columns if c.startswith('demo_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_demos = [c for c in demo_cols if df[c].sum() >= min_count]
dropped = set(demo_cols) - set(valid_demos)
if dropped:
    print(f"Dropping {len(dropped)} rare demographics from regression: {sorted(dropped)}")

X = df[valid_demos].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['drop_rate_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'demographic': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("demographic != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

## Genre, Theme, Demographic vs. Forum Z-score (Multiple Regression)

In [ ]:
genre_cols = [c for c in df.columns if c.startswith('genre_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_genres = [c for c in genre_cols if df[c].sum() >= min_count]
dropped = set(genre_cols) - set(valid_genres)
if dropped:
    print(f"Dropping {len(dropped)} rare genres from regression: {sorted(dropped)}")

X = df[valid_genres].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['forum_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'genre': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("genre != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

In [ ]:
theme_cols = [c for c in df.columns if c.startswith('theme_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_themes = [c for c in theme_cols if df[c].sum() >= min_count]
dropped = set(theme_cols) - set(valid_themes)
if dropped:
    print(f"Dropping {len(dropped)} rare themes from regression: {sorted(dropped)}")

X = df[valid_themes].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['forum_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'theme': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("theme != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

In [ ]:
demo_cols = [c for c in df.columns if c.startswith('demo_')]

# drop genres that are too rare to estimate reliably (mirrors your bucket_rare logic)
min_count = 50
valid_demos = [c for c in demo_cols if df[c].sum() >= min_count]
dropped = set(demo_cols) - set(valid_demos)
if dropped:
    print(f"Dropping {len(dropped)} rare demographics from regression: {sorted(dropped)}")

X = df[valid_demos].copy()
X = sm.add_constant(X)  # intercept = baseline z-score for "no genres present" (rarely meaningful alone, but needed for coefficients to be interpretable as offsets)
y = df['forum_z']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

results = pd.DataFrame({
    'demographic': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("demographic != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
results

## Rating, Sequel vs. Score Z-score (ANOVA + Tukey, T-test)

In [ ]:
# one-way ANOVA
groups = [df.loc[df['rating'] == r, 'score_z'].dropna() for r in df['rating'].unique()]
f_stat, p_val = stats.f_oneway(*groups)
print(f"F={f_stat:.3f}, p={p_val:.4f}")

# Tukey post-hoc
tukey = pairwise_tukeyhsd(endog=df['score_z'], groups=df['rating'], alpha=0.05)
print(tukey)

In [ ]:
# seq_yes = df.loc[df['has_prequel_score'] == 1, 'score_z'].dropna()
# seq_no  = df.loc[df['has_prequel_score'] == 0, 'score_z'].dropna()

# t_stat, p_val = stats.ttest_ind(seq_yes, seq_no, equal_var=False)  # Welch's, safer default
# print(f"t={t_stat:.3f}, p={p_val:.4f}, mean_diff={seq_yes.mean() - seq_no.mean():.3f}")

We want to also find the partial effects of ratings and sequels taking into account genres, themes, and demographics since they are likely related (e.g. demographics and age ratings definitely correlate). By "partial effects," I mean that if we keep everything else constant, what effect does it have on the score z-score?

In [ ]:
rating_dummies = pd.get_dummies(df['rating'], prefix='rating', drop_first=True)
X = pd.concat([df[valid_genres], df[valid_themes], df[valid_demos], rating_dummies, df[['has_prequel_score', 'has_prequel_members']]], axis=1)
X = X.astype(float)
X = sm.add_constant(X)

model = sm.OLS(df['score_z'], X, missing='drop').fit(cov_type='HC3')
print(model.summary())

In [ ]:
results = pd.DataFrame({
    'feature': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("feature != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
print(results.loc[results['significant']==True, 'coef'].head(10))
print(results.loc[results['significant']==True, 'coef'].tail(10))

The above shows the top 10 and bottom 10 coefficients for features that yielded statistically significant results $(p < 0.05)$.

## Other Metrics

For the other metrics, we will go straight to partial effects.

In [ ]:
rating_dummies = pd.get_dummies(df['rating'], prefix='rating', drop_first=True)
X = pd.concat([df[valid_genres], df[valid_themes], df[valid_demos], rating_dummies, df[['has_prequel_score', 'has_prequel_members']]], axis=1)
X = X.astype(float)
X = sm.add_constant(X)

model = sm.OLS(df['wc_z'], X, missing='drop').fit(cov_type='HC3')
print(model.summary())

In [ ]:
results = pd.DataFrame({
    'feature': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("feature != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
print(results.loc[results['significant']==True, 'coef'].head(10))
print(results.loc[results['significant']==True, 'coef'].tail(10))

In [ ]:
rating_dummies = pd.get_dummies(df['rating'], prefix='rating', drop_first=True)
X = pd.concat([df[valid_genres], df[valid_themes], df[valid_demos], rating_dummies, df[['has_prequel_score', 'has_prequel_members']]], axis=1)
X = X.astype(float)
X = sm.add_constant(X)

model = sm.OLS(df['favorites_z'], X, missing='drop').fit(cov_type='HC3')
print(model.summary())

In [ ]:
results = pd.DataFrame({
    'feature': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("feature != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
print(results.loc[results['significant']==True, 'coef'].head(10))
print(results.loc[results['significant']==True, 'coef'].tail(10))

In [ ]:
rating_dummies = pd.get_dummies(df['rating'], prefix='rating', drop_first=True)
X = pd.concat([df[valid_genres], df[valid_themes], df[valid_demos], rating_dummies, df[['has_prequel_score', 'has_prequel_members']]], axis=1)
X = X.astype(float)
X = sm.add_constant(X)

model = sm.OLS(df['drop_rate_z'], X, missing='drop').fit(cov_type='HC3')
print(model.summary())

In [ ]:
results = pd.DataFrame({
    'feature': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("feature != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
print(results.loc[results['significant']==True, 'coef'].head(10))
print(results.loc[results['significant']==True, 'coef'].tail(10))

In [ ]:
rating_dummies = pd.get_dummies(df['rating'], prefix='rating', drop_first=True)
X = pd.concat([df[valid_genres], df[valid_themes], df[valid_demos], rating_dummies, df[['has_prequel_score', 'has_prequel_members']]], axis=1)
X = X.astype(float)
X = sm.add_constant(X)

model = sm.OLS(df['forum_z'], X, missing='drop').fit(cov_type='HC3')
print(model.summary())

In [ ]:
results = pd.DataFrame({
    'feature': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("feature != 'const'")

# FDR correction across all genre coefficients (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
results['pval_fdr'] = multipletests(results['pval'], method='fdr_bh')[1]
results['significant'] = results['pval_fdr'] < 0.05

results = results.sort_values('coef', ascending=False)
print(results.loc[results['significant']==True, 'coef'].head(10))
print(results.loc[results['significant']==True, 'coef'].tail(10))

NOTE: has_prequel_members and has_prequel_score are giving negative coeffs for some reason. Double check this.